# Qwen2.5-VL eval — Claude apples-to-apples (inference only)

Streamlined notebook for the head-to-head. **No training, no data prep.** Just:
1. Install deps (~2 min, requires kernel restart)
2. Upload `ttb_sft_qwen2_5_vl_7b.zip` from your Mac (the trained adapter, 146 MB)
3. Upload `test/eval/data/cola_sample.csv` from your Mac (the ground-truth CSV)
4. Load Qwen-7B base + apply your LoRA adapter (~5 min — first-time HF download)
5. Run extraction on the 90 Claude-test-set images (~10 min on A100)
6. Auto-downloads `qwen_outputs.jsonl` to your Mac

On your Mac after the download:
```
make eval-replay-qwen   # side-by-side report
```

**Runtime:** A100 + High-RAM (best) or T4 (works, slower).

## 0. Install dependencies

---

> ### 🚀 How to run this notebook
>
> **Two clicks of Run all:**
> 1. **Runtime → Run all** (1st time). Cell 2 installs deps (~2 min) then auto-kills the kernel. Red "session crashed" banner is expected.
> 2. **Runtime → Run all** (2nd time). Cell 2 is a no-op; everything else runs end-to-end.

---

In [ ]:
# Inference-only stack. Same pinned versions as the training notebook so
# the adapter loads correctly.
import importlib.util, subprocess, sys, os


def _have(mod): return importlib.util.find_spec(mod) is not None
def _numpy_ok():
    try:
        import numpy
        p = numpy.__version__.split(".")
        return int(p[0]) == 2 and int(p[1]) < 2
    except Exception:
        return False


def _pinned(mod, want):
    try:
        return __import__(mod).__version__ == want
    except Exception:
        return False

_required = ["peft", "bitsandbytes", "accelerate", "unsloth", "PIL"]
_ready = _numpy_ok() and _pinned("transformers", "4.48.3") and all(_have(m) for m in _required)

if _ready:
    import numpy, transformers
    print(f"✓ Set up — numpy {numpy.__version__}, transformers {transformers.__version__}. Continuing...")
else:
    print("⏳ Installing dependencies (~2 min). Kernel will auto-restart at end.")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--upgrade", "pip"])
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--upgrade",
        "unsloth", "unsloth_zoo",
        "transformers==4.48.3",   # has Qwen2.5-VL, before the 4.49 config regression
        "accelerate>=0.30", "peft<0.14", "bitsandbytes>=0.44", "trl<0.10",
        "protobuf>=5.27",         # transformers 4.48 needs protobuf 5.x for runtime_version
        "pillow", "tqdm"])
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
        "--force-reinstall", "--no-deps", "numpy>=2.0,<2.2"])
    print()
    print("=" * 70)
    print("✅ Install complete. Auto-killing kernel so numpy 2.1 ABI takes effect.")
    print("   Wait for the 'session crashed' banner, then Runtime → Run all AGAIN.")
    print("=" * 70)
    os.kill(os.getpid(), 9)

## 1. Upload the trained adapter

In [ ]:
import zipfile
from pathlib import Path

ADAPTER_ROOT = Path("/content/adapter_bundle")
ADAPTER_DIR = ADAPTER_ROOT / "adapter"

if ADAPTER_DIR.exists() and (ADAPTER_DIR / "adapter_config.json").exists():
    # Idempotent: already extracted (e.g. from a previous run before a kernel restart).
    # /content/ persists across kernel restarts within the same Colab runtime VM.
    print(f"✓ Adapter already extracted at {ADAPTER_DIR} — skipping upload.")
else:
    from google.colab import files
    print("Upload ttb_sft_qwen2_5_vl_7b.zip from your Mac's ~/Downloads:")
    up = files.upload()
    zip_name = next(iter(up))
    zip_path = Path("/content") / zip_name
    zip_path.write_bytes(up[zip_name])
    ADAPTER_ROOT.mkdir(exist_ok=True)
    with zipfile.ZipFile(zip_path, "r") as z:
        z.extractall(ADAPTER_ROOT)
    assert ADAPTER_DIR.exists(), "adapter/ not found in zip"
    assert (ADAPTER_DIR / "adapter_config.json").exists(), "adapter_config.json missing"
    print(f"✓ Adapter unzipped: {ADAPTER_DIR}")

print(f"  contents: {sorted(p.name for p in ADAPTER_DIR.iterdir())}")

## 2. Load Qwen2.5-VL base + apply the LoRA adapter

This downloads `unsloth/Qwen2.5-VL-7B-Instruct-bnb-4bit` from HF (~5 GB, first-time only) and applies your trained LoRA. ~5 min on first run; instant if HF cached.

In [ ]:
import torch
from unsloth import FastVisionModel
from peft import PeftModel

print("Loading Qwen2.5-VL base via Unsloth (matches the training-time loader)...")
base, tokenizer = FastVisionModel.from_pretrained(
    "unsloth/Qwen2.5-VL-7B-Instruct-bnb-4bit",
    load_in_4bit=True,
    use_gradient_checkpointing=False,
)
print(f"Applying LoRA adapter from {ADAPTER_DIR}...")
model = PeftModel.from_pretrained(base, str(ADAPTER_DIR))
FastVisionModel.for_inference(model)
processor = tokenizer    # Unsloth returns the multimodal processor as `tokenizer`
print("✅ Model loaded and ready for inference.")

## 3. Upload cola_sample.csv (the ground-truth file)

In [ ]:
import csv, io
CSV_PATH = Path("/content/cola_sample.csv")

if CSV_PATH.exists():
    csv_text = CSV_PATH.read_text()
    rows = list(csv.DictReader(io.StringIO(csv_text)))
    print(f"✓ Reused cached CSV ({CSV_PATH}): {len(rows)} rows")
else:
    from google.colab import files
    print("Upload test/eval/data/cola_sample.csv from your Mac:")
    up = files.upload()
    csv_name = next(iter(up))
    csv_text = up[csv_name].decode("utf-8")
    CSV_PATH.write_text(csv_text)
    rows = list(csv.DictReader(io.StringIO(csv_text)))
    print(f"✓ Loaded {len(rows)} rows from {csv_name} (cached to {CSV_PATH})")

## 4. Download the test images from the COLA Cloud CDN (~1 min)

In [ ]:
import urllib.request, concurrent.futures
EVAL_DIR = Path("/content/qwen_eval"); EVAL_DIR.mkdir(exist_ok=True)
IMG_DIR = EVAL_DIR / "images"; IMG_DIR.mkdir(exist_ok=True)
CDN = "https://dyuie4zgfxmt6.cloudfront.net/{}.webp"

def _dl(row):
    image_id = row["ttb_image_id"]
    dest = IMG_DIR / f"{image_id}.webp"
    if dest.exists() and dest.stat().st_size > 0:
        return row, True
    try:
        req = urllib.request.Request(CDN.format(image_id), headers={"User-Agent": "ttb-eval/0.1"})
        with urllib.request.urlopen(req, timeout=20) as r:
            dest.write_bytes(r.read())
        return row, True
    except Exception as e:
        print(f"  download fail {image_id}: {e}")
        return row, False

downloaded = []
with concurrent.futures.ThreadPoolExecutor(max_workers=16) as ex:
    for row, ok in ex.map(_dl, rows):
        if ok:
            row["_image_path"] = str(IMG_DIR / f"{row['ttb_image_id']}.webp")
            downloaded.append(row)
print(f"✓ Downloaded {len(downloaded)} / {len(rows)} images")

## 5. Run Qwen extraction on each image (~10 min on A100)

In [ ]:
import json, re, time, statistics
from PIL import Image
from collections import Counter

SYSTEM_PROMPT = (
    "You are a careful transcription assistant for U.S. TTB alcohol label review. "
    "Given ONE label panel image and the beverage type, READ what is printed and "
    "RETURN ONLY a JSON object matching the schema below. Do NOT decide compliance.\n\n"
    "If a field is not clearly visible, OMIT it from the fields object. Never guess; "
    "never substitute deposit codes (e.g. \"CA CRV\"), NOM IDs, or barcodes. Transcribe "
    "the Government Warning EXACTLY as printed (preserve case, punctuation, errors).\n\n"
    "Schema: {fields: {<field name>: {value, confidence}}, government_warning: "
    "{present, detected_text, casing_all_caps, heading_bold, body_bold, approx_font_mm, "
    "contrast_ok, separate_and_apart}, image_quality: {score, legible, note}}"
)

def _qwen_extract(image_path, beverage_type):
    img = Image.open(image_path).convert("RGB")
    msgs = [
        {"role": "system", "content": [{"type": "text", "text": SYSTEM_PROMPT}]},
        {"role": "user",   "content": [
            {"type": "image", "image": img},
            {"type": "text",  "text": f"Beverage type: {beverage_type}. Extract per the schema."},
        ]},
    ]
    text = processor.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inputs = processor(text=text, images=[img], return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=512, do_sample=False)
    decoded = processor.batch_decode(out[:, inputs["input_ids"].shape[1]:], skip_special_tokens=True)[0]
    s, e = decoded.find("{"), decoded.rfind("}")
    if s == -1 or e == -1:
        return None, decoded
    try:
        return json.loads(decoded[s:e+1]), decoded
    except json.JSONDecodeError:
        cleaned = re.sub(r",(\s*[}\]])", r"\1", decoded[s:e+1])
        try: return json.loads(cleaned), decoded
        except Exception: return None, decoded

qwen_outputs_jsonl = []
n_parsed = n_unparseable = 0
latencies = []

print(f"\nExtracting on {len(downloaded)} images...")
for i, row in enumerate(downloaded, 1):
    bev = (row.get("beverage_type") or "wine").lower()
    t0 = time.time()
    pred, raw_text = _qwen_extract(row["_image_path"], bev)
    dt = time.time() - t0
    latencies.append(dt)
    qwen_outputs_jsonl.append({
        "image_filename": row.get("image_filename") or (row["ttb_image_id"] + ".webp"),
        "ttb_image_id":   row["ttb_image_id"],
        "beverage_type":  bev,
        "raw_output":     raw_text,
        "parsed_output":  pred,
        "latency_sec":    round(dt, 3),
    })
    if pred is None:
        n_unparseable += 1
    else:
        n_parsed += 1
    if i % 10 == 0:
        print(f"  [{i}/{len(downloaded)}] parsed={n_parsed} unparseable={n_unparseable}")

print(f"\nDone. Parsed: {n_parsed} / Unparseable: {n_unparseable}")
print(f"Latency mean / p95: {statistics.mean(latencies):.2f}s / {sorted(latencies)[int(len(latencies)*0.95)]:.2f}s")

## 6. Save + auto-download the JSONL fixture for local replay

In [ ]:
jsonl_path = EVAL_DIR / "qwen_outputs.jsonl"
with open(jsonl_path, "w") as f:
    for rec in qwen_outputs_jsonl:
        f.write(json.dumps(rec) + "\n")
print(f"✓ Wrote {jsonl_path} ({len(qwen_outputs_jsonl)} rows)")

files.download(str(jsonl_path))
print()
print("On your Mac next:")
print("  make eval-replay-qwen")
print("  # reads ~/Downloads/qwen_outputs.jsonl, writes test/eval/qwen_report.json,")
print("  # prints side-by-side vs Claude's existing test/eval/report.json")